# Monte Carlo VaR with Financial Modeling Prep

This notebook pulls historical prices from Financial Modeling Prep (FMP), builds a sample
portfolio, and estimates Value-at-Risk (VaR) using a Monte Carlo simulation.


## Prerequisites
- Set your FMP API key as an environment variable: `FMP_API_KEY`
- Install dependencies: `pip install pandas numpy requests matplotlib`

You can get an API key at https://financialmodelingprep.com/developer/docs


In [ ]:
import os
import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
pd.set_option("display.width", 120)

FMP_API_KEY = os.getenv("FMP_API_KEY", "").strip()
if not FMP_API_KEY:
    raise RuntimeError("Set FMP_API_KEY in your environment before running this notebook.")


## Download historical prices
We will download adjusted close prices for a small sample portfolio.


In [ ]:
TICKERS = ["AAPL", "MSFT", "NVDA", "JPM"]
END_DATE = datetime.utcnow().date()
START_DATE = END_DATE - timedelta(days=365 * 2)

def fetch_prices_fmp(ticker, start_date, end_date, api_key):
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}"
    params = {
        "from": start_date.strftime("%Y-%m-%d"),
        "to": end_date.strftime("%Y-%m-%d"),
        "serietype": "line",
        "apikey": api_key,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if "historical" not in data:
        raise ValueError(f"No historical data for {ticker}: {data}")
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")["close"].sort_index()
    df.name = ticker
    return df

price_series = [fetch_prices_fmp(t, START_DATE, END_DATE, FMP_API_KEY) for t in TICKERS]
prices = pd.concat(price_series, axis=1).dropna()
prices.tail()


In [ ]:
prices.plot(figsize=(10, 4), title="Adjusted Close Prices")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()


## Build a sample portfolio
Define portfolio weights (must sum to 1.0). We'll use a simple equal-weight portfolio.


In [ ]:
weights = np.array([1.0 / len(TICKERS)] * len(TICKERS))
weights


## Compute returns
We will use daily log returns for the simulation.


In [ ]:
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.head()


## Monte Carlo VaR
We simulate correlated returns using the historical mean and covariance.
Change `NUM_SIMULATIONS`, `HORIZON_DAYS`, and `CONFIDENCE` as needed.


In [ ]:
NUM_SIMULATIONS = 10000
HORIZON_DAYS = 10
CONFIDENCE = 0.95

mu = log_returns.mean().values
cov = log_returns.cov().values

# Cholesky for correlated random draws
chol = np.linalg.cholesky(cov)

# Simulate daily log returns and aggregate over horizon
z = np.random.standard_normal((NUM_SIMULATIONS, HORIZON_DAYS, len(TICKERS)))
daily_sim = z @ chol.T + mu

# Portfolio log return per simulation over the horizon
port_log_returns = (daily_sim @ weights).sum(axis=1)

# Convert log returns to simple returns
port_simple_returns = np.exp(port_log_returns) - 1.0

var_level = 1.0 - CONFIDENCE
var_value = np.quantile(port_simple_returns, var_level)

print(f"{int(CONFIDENCE*100)}% {HORIZON_DAYS}-day VaR: {abs(var_value):.2%}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(port_simple_returns, bins=60, alpha=0.7)
plt.axvline(var_value, color="red", linestyle="--", label="VaR")
plt.title("Simulated Portfolio Returns")
plt.xlabel("Return")
plt.ylabel("Frequency")
plt.legend()
plt.show()


## Optional: Export results
Save your simulated return distribution for downstream analysis.


In [ ]:
output_path = "/home/ubuntu/monte-carlo-GPU-CUDA/results/var_simulated_returns.csv"
pd.Series(port_simple_returns, name="simulated_return").to_csv(output_path, index=False)
print(f"Saved simulated returns to {output_path}")
